# Domain 4: Evaluation, Testing & Optimization
**Exam**: Claude Certified Architect – Professional (CCAR-P)
**Weighting**: 16% of exam
**Depth**: deep

## Domain Overview

This domain covers defining evaluation metrics (accuracy, latency, cost, safety, security),
designing evaluation datasets and mixed-methodology test frameworks, conducting A/B testing,
diagnosing system issues (prompt failure, hallucinations, model mismatch), optimizing token/latency/
cost tradeoffs, and monitoring via logging/observability tools.

**Why it matters**: professional architects are expected to prove a system works and keep proving it
as it evolves, not just ship it once.

**Key concepts to know**: multi-dimensional evaluation metrics, evaluation dataset design, A/B
testing methodology, root-cause diagnosis, and production monitoring.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Define evaluation metrics across accuracy, latency, cost, safety, and security

A complete evaluation framework tracks multiple dimensions simultaneously — optimizing
only for accuracy can silently regress cost or latency. Safety and security metrics (e.g., guardrail
bypass rate, PII leakage rate) deserve the same rigor as accuracy metrics.

In [ ]:
def evaluation_scorecard(accuracy: float, p95_latency_ms: int, cost_per_request: float, guardrail_bypass_rate: float) -> dict:
    return {
        "accuracy": accuracy,
        "p95_latency_ms": p95_latency_ms,
        "cost_per_request": cost_per_request,
        "guardrail_bypass_rate": guardrail_bypass_rate,
        "release_recommended": accuracy > 0.9 and guardrail_bypass_rate < 0.01,
    }

print(evaluation_scorecard(0.94, 1800, 0.012, 0.003))

### Design evaluation datasets and test frameworks using mixed methodologies

Good eval datasets combine real production examples (representative distribution),
adversarial/edge cases (stress the boundaries), and expert-labeled golden examples (ground truth for
scoring). Mixed methodology also means combining automated scoring with periodic human review.

In [ ]:
eval_dataset_composition = {
    "real production samples": 0.6,
    "adversarial/edge cases": 0.25,
    "expert-labeled golden examples": 0.15,
}
assert abs(sum(eval_dataset_composition.values()) - 1.0) < 1e-9
print(eval_dataset_composition)

### Conduct A/B testing and iterative improvements

A/B testing compares a candidate prompt/model/config against the current production
version on a live (or held-out) traffic split, measuring the same scorecard metrics. Statistical
significance and a large-enough sample matter — a 2% accuracy delta on 50 samples isn't reliable
evidence.

In [ ]:
def is_ab_test_conclusive(sample_size: int, observed_delta_pct: float, min_sample_size: int = 500) -> bool:
    return sample_size >= min_sample_size and abs(observed_delta_pct) >= 2.0

print(is_ab_test_conclusive(50, 2.0))
print(is_ab_test_conclusive(2000, 2.5))

### Diagnose system issues (prompt failure, hallucinations, model mismatch)

Root-cause diagnosis should separate: prompt failures (ambiguous/missing instructions),
hallucinations (fabricated specifics), and model mismatch (using a tier unsuited to the task's actual
complexity). Each has a different fix — more context, stricter output constraints, or a different
model tier, respectively.

In [ ]:
def diagnose_issue(symptom: str) -> str:
    if "made up a statistic" in symptom or "fake citation" in symptom:
        return "hallucination - add grounding/retrieval, require citations, add validation"
    if "inconsistent format" in symptom or "sometimes ignores instructions" in symptom:
        return "prompt failure - clarify instructions, add few-shot examples"
    if "fails on complex multi-step reasoning" in symptom:
        return "model mismatch - upgrade model tier or add chain-of-thought"
    return "needs further investigation"

print(diagnose_issue("made up a statistic"))
print(diagnose_issue("fails on complex multi-step reasoning"))

### Optimize token usage, latency, and cost-performance trade-offs; monitor via logging

Optimization should be driven by observed data, not guesses: token/latency/cost dashboards
segmented by request type reveal where the biggest wins are. A single aggregate cost number can hide
a small subset of expensive request types dominating the total.

In [ ]:
cost_by_request_type = {"simple_faq": 0.001, "document_summary": 0.02, "multi_agent_research": 0.35}
volume_by_request_type = {"simple_faq": 50000, "document_summary": 4000, "multi_agent_research": 300}

total_cost = {k: cost_by_request_type[k] * volume_by_request_type[k] for k in cost_by_request_type}
print(total_cost)
print("Highest total-cost driver:", max(total_cost, key=total_cost.get))

## Mini Hands-On Project

**Covers**: all Domain 4 sub-objectives.

**Scenario**: Evaluate and optimize a document-summary feature that has drawn complaints about
occasional fabricated numbers.

### Step 1: Score the current system

In [ ]:
print(evaluation_scorecard(accuracy=0.88, p95_latency_ms=2100, cost_per_request=0.02, guardrail_bypass_rate=0.04))

### Step 2: Diagnose the fabricated-numbers complaint

In [ ]:
print(diagnose_issue("made up a statistic"))

### Step 3: Design an A/B test for a grounding-focused fix and check conclusiveness

In [ ]:
print(is_ab_test_conclusive(sample_size=1200, observed_delta_pct=3.5))
print(total_cost)  # identify if document_summary is a major cost driver worth optimizing too

## Lab Exercises

### Exercise 1: Multi-dimensional scorecard review

**Objective**: Evaluate 3 candidate configs and pick the best release candidate.

**Produce**: 3 scorecards and a release decision.

In [ ]:
candidates = [
    ("A: fast/cheap", 0.86, 900, 0.005, 0.02),
    ("B: balanced", 0.93, 1600, 0.015, 0.005),
    ("C: max accuracy", 0.97, 3200, 0.08, 0.001),
]
# TODO: call evaluation_scorecard for each and decide which to release given a 2s latency budget
for name, acc, lat, cost, bypass in candidates:
    pass

# Expected output: Candidate B likely wins given the 2s budget; C exceeds latency, A fails the accuracy/guardrail bar.

In [ ]:
# STRETCH GOAL: propose a hybrid approach: route by request complexity to different candidates.
routing_plan = "..."  # TODO

# Expected output: A routing plan sending simple requests to A/B and only complex/high-stakes ones to C.

### Exercise 2: Eval dataset composition audit

**Objective**: Audit an eval dataset missing adversarial cases and fix the composition.

**Produce**: A corrected composition and 2 example adversarial cases to add.

In [ ]:
current_composition = {"real production samples": 0.95, "adversarial/edge cases": 0.0, "expert-labeled golden examples": 0.05}
# TODO: propose a corrected composition and 2 concrete adversarial test cases relevant to a claims-assistant
corrected_composition = {}  # TODO
adversarial_examples = []  # TODO
print(corrected_composition, adversarial_examples)

# Expected output: A corrected composition including a meaningful adversarial share, plus 2 concrete, relevant adversarial examples.

In [ ]:
# STRETCH GOAL: propose a process for continuously adding new adversarial cases discovered in production.
process = "..."  # TODO

# Expected output: A described process (e.g., flagging escalations as candidate adversarial cases for the eval set).

### Exercise 3: A/B test sample size planning

**Objective**: Determine if 3 different A/B test results are conclusive.

**Produce**: 3 conclusiveness determinations.

In [ ]:
tests = [
    (300, 3.0),
    (1500, 1.0),
    (5000, 2.2),
]
# TODO: call is_ab_test_conclusive for each and print results
for size, delta in tests:
    pass

# Expected output: Test 1 not conclusive (sample too small), test 2 not conclusive (delta too small), test 3 conclusive.

In [ ]:
# STRETCH GOAL: for the non-conclusive tests, recommend a specific next step (larger sample vs. bigger effect needed).
recommendations = []  # TODO

# Expected output: Distinct, correct recommendations per non-conclusive case.

### Exercise 4: Root-cause diagnosis batch

**Objective**: Diagnose 4 new reported symptoms.

**Produce**: 4 diagnoses with recommended fixes.

In [ ]:
symptoms = [
    "Cites a page number that doesn't exist in the source document",
    "Sometimes responds in the wrong format even though the prompt specifies JSON",
    "Struggles to correctly resolve a 6-step legal liability chain",
    "Occasionally very slow, no clear pattern",
]
# TODO: call diagnose_issue for each (adapt reasoning for the 4th, which isn't covered by the existing function)
for s in symptoms:
    pass

# Expected output: Correct diagnoses for symptoms 1-3; symptom 4 recognized as needing further latency-specific investigation (not covered by the 3 categories).

In [ ]:
# STRETCH GOAL: extend diagnose_issue to add a 4th category for intermittent latency issues,
# suggesting checking for retries, rate limiting, or cold starts.
def diagnose_issue_v2(symptom: str) -> str:
    pass

# Expected output: An extended function correctly categorizing the latency symptom with a relevant suggested fix.

### Exercise 5: Cost driver deep-dive

**Objective**: Given cost_by_request_type and volume, propose one concrete optimization targeting the biggest driver.

**Produce**: An identified driver and a specific optimization proposal.

In [ ]:
# TODO: identify the biggest cost driver from total_cost (reuse from the concept cell) and propose
# a specific optimization (e.g., route to a cheaper model, add caching, reduce agent hops)
biggest_driver = max(total_cost, key=total_cost.get)
optimization_proposal = "..."  # TODO
print(biggest_driver, optimization_proposal)

# Expected output: Correctly identifies multi_agent_research as the biggest driver, with a specific, plausible optimization proposal.

In [ ]:
# STRETCH GOAL: estimate the cost savings if the optimization cuts that segment's cost_per_request by 30%.
projected_savings = total_cost[biggest_driver] * 0.30
print(projected_savings)

# Expected output: A concrete projected dollar savings figure from the proposed optimization.

## Self-Assessment: Evaluation, Testing & Optimization

Answer each question, then check the Answers cell below.


**Q1.** Why is optimizing only for accuracy a risky evaluation strategy?

**Q2.** What three types of examples should a well-designed evaluation dataset combine?

**Q3.** Why is a 2% accuracy improvement on only 50 samples not reliable evidence for an A/B test decision?

**Q4.** What distinguishes a 'hallucination' from a 'model mismatch' as a root cause?

**Q5.** Why can an aggregate cost number mislead optimization efforts?

## Answers


**A1.** Because it can silently allow regressions in cost, latency, or safety while accuracy alone looks good.

**A2.** Real production samples, adversarial/edge cases, and expert-labeled golden examples.

**A3.** Because the sample size is too small to be statistically reliable — the observed difference could easily be noise.

**A4.** A hallucination is a fabricated specific detail regardless of task complexity; a model mismatch is using a model tier unsuited to the task's actual reasoning complexity.

**A5.** Because it can hide a small subset of expensive request types that dominate total spend, which segment-level analysis reveals but an aggregate number does not.

---
**Next**: [Domain 5: Governance, Safety & Risk Management](./05-governance-safety-and-risk-management.ipynb)
**Back**: [README](./README.md)